<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/Python-2026/Practice/Matplotlib_Practica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📖 Раздел 0. Откуда взять данные

Прежде чем построить первый график, нужно ответить на вопрос, который студенты часто стесняются задавать: «А где взять `bookings.csv`?». В реальной работе данные приходят из базы, CRM, API туроператора или файла от заказчика. В учебной — их можно **найти готовыми**, **сгенерировать самому** или **скомбинировать**: взять реальную структуру и заменить значения синтетическими. Разберём все три пути, потому что каждый важен в своей ситуации.

**Три источника данных — и когда какой выбирать.**

**Готовый датасет** — когда нужно «как в жизни»: реальные распределения, пропуски, выбросы, дисбаланс классов. Плюс: не нужно ничего придумывать, данные уже «грязные» — как в продакшене. Минус: нужно найти, скачать, разобраться со структурой, а иногда и с лицензией.

**Синтетические данные** — когда нужен **контроль**: вы точно знаете, что должно получиться на графике, можете воспроизвести результат с `seed=42`, задать нужный дисбаланс или добавить ровно 3 выброса. Плюс: полный контроль, воспроизводимость, не зависите от внешних источников. Минус: данные «игрушечные», могут не отражать реальных сложностей.

**Гибрид** — взять реальную структуру (например, из Kaggle или Hugging Face), а значения заменить или дополнить синтетическими. Это лучший вариант для курса: структура реалистична, а свойства данных (пропуски, шум, дисбаланс) — под контролем.

---

## 0.1. Готовые датасеты: где искать

### Встроенные в Seaborn

Seaborn поставляется с набором учебных датасетов, которые загружаются одной строкой через `sns.load_dataset(name)`. Это лучший старт: они маленькие, чистые и с понятными колонками. Посмотреть полный список можно через `sns.get_dataset_names()`. Типичный набор включает `tips`, `titanic`, `flights`, `penguins`, `iris`, `diamonds`, `fmri`, `exercise`, `healthexp` и другие.

```python
import seaborn as sns
import pandas as pd

print(sns.get_dataset_names())          # список доступных датасетов

tips = sns.load_dataset("tips")          # загружает DataFrame
flights = sns.load_dataset("flights")    # пассажиры по месяцам и годам
titanic = sns.load_dataset("titanic")    # категориальные признаки, пропуски

# Сохранить локально, чтобы задачи работали без интернета
tips.to_csv("tips.csv", index=False)
```

Для курса по турагентству особенно полезны `flights` (сезонность спроса — аналог продаж туров по месяцам) и `titanic` (категориальные признаки, пропуски, бинарная целевая — учебная модель для прогноза оттока клиентов). Задача 9 (гистограмма с правилом Фридмана-Диакониса) отлично работает на `tips`: колонка `total_bill` даёт реалистичное распределение с правым хвостом.

**Важно.** `sns.load_dataset` требует интернет при первом вызове. В офлайн-среде скачайте датасет заранее и сохраните как CSV через `to_csv`, как показано выше.

### Встроенные в scikit-learn

Для ML-задач (ROC, PR, learning curves, feature importance) есть генераторы и загрузчики. `make_classification` создаёт синтетический датасет для классификации с контролем числа признаков, информативности и **дисбаланса классов** — именно то, что нужно для задачи 38 (ROC vs PR). `make_regression` — для регрессии, `make_blobs` — для кластеризации. Загрузчики `load_iris`, `load_digits`, `fetch_california_housing` возвращают готовые данные.

```python
from sklearn.datasets import make_classification
import pandas as pd

X, y = make_classification(
    n_samples=5000, n_features=10, n_informative=5,
    n_classes=2, weights=[0.95, 0.05],   # 95% / 5% — дисбаланс
    random_state=42,
)
df = pd.DataFrame(X, columns=[f"f{i}" for i in range(X.shape[1])])
df["churn"] = y
df.to_csv("tourists_ml.csv", index=False)
```

Здесь `weights=[0.95, 0.05]` — ключевой параметр: он создаёт **дисбаланс**, при котором ROC-кривая может «обманывать», а PR-кривая показывает реальную картину. Без этого задача 38 теряет смысл.

### Kaggle через `kagglehub`

`kagglehub` — официальная библиотека Kaggle. Она **скачивает файлы датасета** в локальный кэш и возвращает **путь к папке**, а не сам файл. Дальше вы читаете CSV через Pandas, как обычно.

```python
import kagglehub
import pandas as pd
from pathlib import Path

# Скачивает датасет в кэш и возвращает путь к папке
path = kagglehub.dataset_download("author/hotel-booking-demand")
print("Path to dataset files:", path)

# Дальше — обычный pandas: путь + имя файла
df = pd.read_csv(Path(path) / "hotel_bookings.csv")

# Скопировать в рабочую папку, чтобы задачи работали без f"{path}/..."
df.to_csv("bookings.csv", index=False)
```

**Скачать один конкретный файл** (когда архив большой, а нужен только CSV): `kagglehub.dataset_download("author/dataset", path="bookings.csv")`. **Зафиксировать версию:** `version=1` в вызове — иначе при обновлении датасета автором ваши графики изменятся.

**Авторизация.** Нужен `kaggle.json` в `~/.kaggle/` или переменные окружения `KAGGLE_USERNAME` / `KAGGLE_KEY`. В Kaggle Notebooks и Colab авторизация уже настроена.

### Hugging Face через `datasets`

Hugging Face работает иначе: `load_dataset` возвращает **объект `Dataset`** (или `DatasetDict` с разбивками `train`/`validation`/`test`), а не путь к файлу. Конвертация в Pandas — через `.to_pandas()`.

```python
from datasets import load_dataset
import pandas as pd

# Загружает датасет как Dataset
dataset = load_dataset("username/dataset-name", split="train")
df = dataset.to_pandas()                  # конвертация в DataFrame

# Или через DatasetDict, если есть разбивки
ds = load_dataset("username/dataset-name")
df_train = ds["train"].to_pandas()
df_test = ds["test"].to_pandas()

df_train.to_csv("bookings.csv", index=False)   # сохраняем локально
```

**Плюс Hugging Face:** для публичных датасетов не нужна авторизация; многие табличные датасеты хранятся в Parquet и читаются быстро; `load_dataset` сразу даёт разбивки. **Минус:** синтаксис отличается от `pd.read_csv`, поэтому в курсе удобнее **один раз** сконвертировать в локальный CSV — и дальше все задачи работают как раньше.

### Открытые репозитории и поиск

Помимо Kaggle и Hugging Face, есть UCI Machine Learning Repository, Google Dataset Search, открытые порталы (`data.gov`, EU Open Data Portal), `pandas-datareader` (финансовые временные ряды), `statsmodels.datasets.get_rdataset` (сотни экономических датасетов). Главное правило: **проверяйте лицензию** перед использованием в коммерческом проекте. Для учебного курса это обычно не проблема, но если курс публикуется — убедитесь, что лицензия позволяет распространение.

---

## 0.2. Синтетические данные: когда реальных нет

Настоящий датасет — это роскошь. В 80% учебных и тестовых сценариев данные **генерируют**, и это правильно: вы контролируете структуру, пропуски, выбросы, распределения и точно знаете, что должно получиться на графике. Три инструмента закрывают почти все потребности.

### NumPy — для числовых данных

`np.random.default_rng(seed)` создаёт генератор без глобального состояния. `rng.normal(loc, scale, size)` — нормальное распределение (возраст, чеки). `rng.uniform(low, high, size)` — равномерное (конверсия, рейтинг). `rng.integers(low, high, size)` — целые (количество туристов). `rng.poisson(lam, size)` — пуассоновское (число броней в день). `rng.choice(categories, size, p=probs)` — категориальное с вероятностями (направления с разной популярностью). `rng.lognormal(mean, sigma, size)` — логнормальное (цены, доходы — асимметричные распределения с правым хвостом, реалистичные для денег).

```python
import numpy as np

rng = np.random.default_rng(42)

ages = rng.integers(18, 75, 1000)                    # возраст
incomes = rng.lognormal(10.5, 0.6, 1000)             # доходы
destinations = rng.choice(
    ["Турция", "Египет", "Италия", "Сочи", "Таиланд"],
    size=1000, p=[0.30, 0.20, 0.15, 0.15, 0.20],
)
```

**Почему логнормальное для цен.** Реальные цены туров асимметричны: большинство в районе 60–150 тыс., но есть «хвост» дорогих. Гауссово распределение дало бы отрицательные цены — классическая ошибка новичка. `lognormal` всегда положительно и имеет правый хвост — как в жизни.

### Faker — для реалистичных строк

`Faker("ru_RU")` генерирует имена (`fake.name()`), email (`fake.email()`), адреса (`fake.address()`), телефоны (`fake.phone_number()`), даты (`fake.date_between(start_date="-2y", end_date="today")`), компании (`fake.company()`). Это критично для визуализаций, где важна **читаемость подписей**: на графике «Турист: Анна Иванова» смотрится лучше, чем «Турист: user_42». Faker особенно полезен для категориальных признаков с высокой кардинальностью — названий отелей, направлений, туроператоров.

```python
from faker import Faker

fake = Faker("ru_RU")
Faker.seed(42)                              # воспроизводимость

names = [fake.name() for _ in range(5)]
emails = [fake.email() for _ in range(5)]
dates = [fake.date_between("-2y", "today") for _ in range(5)]
```

**Важно.** Faker не гарантирует согласованности (имя и email не связаны), но для визуализации это не критично — важна читаемость на графике. Если нужна согласованность (email = транслит имени), используйте транслитерацию и ручную сборку.

### Комбинация — для реалистичных датасетов

Сгенерируйте структуру через NumPy (числа) и Faker (строки), затем добавьте **шум** и **пропуски** вручную — так вы получите датасет, на котором можно отрабатывать очистку и поиск выбросов.

```python
import numpy as np
import pandas as pd
from faker import Faker

rng = np.random.default_rng(42)
fake = Faker("ru_RU")
Faker.seed(42)

n = 1000
destinations = ["Турция", "Египет", "Италия", "Сочи", "Таиланд"]
dest_probs = [0.30, 0.20, 0.15, 0.15, 0.20]

df = pd.DataFrame({
    "booking_id": np.arange(100000, 100000 + n),
    "booking_date": pd.to_datetime(
        rng.integers(0, 730, n), unit="D", origin="2023-01-01"
    ),
    "destination": rng.choice(destinations, n, p=dest_probs),
    "total_price": rng.lognormal(11.5, 0.5, n).round(-2).astype(int),
    "duration_days": rng.choice([3, 5, 7, 10, 14], n),
    "tourist_name": [fake.name() for _ in range(n)],
    "email": [fake.email() if rng.random() > 0.05 else None for _ in range(n)],
})
```

Здесь `rng.lognormal(11.5, 0.5, n)` даёт медиану около 100 тыс. руб. (потому что `exp(11.5) ≈ 98 700`), `round(-2)` округляет до сотен — так цены выглядят реалистичнее. `5%` пропусков в `email` — для задач по очистке.

---

## 0.3. Шум, пропуски и выбросы: готовые рецепты

**Шум** — это искусственное искажение данных, которое делает их «реалистичными» и позволяет проверить устойчивость методов. В учебных датасетах шум нужен по трём причинам: (1) чтобы графики не выглядели «идеально линейными» — реальные данные всегда с разбросом; (2) чтобы проверить, как методы (регрессия, фильтрация) справляются с помехами; (3) чтобы сгенерировать **выбросы** для задач по их обнаружению.

```python
def add_gaussian_noise(y, sigma):
    """Гауссов шум — для облака точек (задача 17)."""
    return y + np.random.normal(0, sigma, size=len(y))

def add_percent_noise(y, percent):
    """Процентный шум — для цен (шум 5% от 100 тыс. и от 1 млн — разные вещи)."""
    return y * (1 + np.random.normal(0, percent, size=len(y)))

def add_outliers(y, n_outliers=3, sigma=10):
    """Выбросы — для задач 17, 27 (RANSAC)."""
    y = y.copy()
    idx = np.random.choice(len(y), n_outliers, replace=False)
    y[idx] += np.random.normal(0, sigma, n_outliers)
    return y

def add_missing(df, column, percent=0.05):
    """Пропуски — для задач 18–21 (dropna, fillna, interpolate)."""
    df = df.copy()
    mask = np.random.random(len(df)) < percent
    df.loc[mask, column] = np.nan
    return df
```

**Разбор.** `add_gaussian_noise` — для задачи 17 (scatter с трендом): маленькая `sigma` даёт почти идеальную линию, большая — размытое облако. `add_percent_noise` — для цен: абсолютный шум не имеет смысла, когда значения различаются в 10 раз. `add_outliers` — для задачи 27 (RANSAC): три точки с большим отклонением, которые «ломают» обычную линейную регрессию. `add_missing` — для задач 18–21: 5% пропусков в `email` или `value`.

**Важно:** все функции возвращают **копию** — не мутируют исходные данные. Это принцип из задачи 56 (антипаттерны: `inplace=True` ломает цепочки).

**Шум в датах** — сдвиньте даты на случайное число дней: `dates + pd.to_timedelta(rng.integers(-2, 3, size), unit="D")` — имитирует «данные пришли с опозданием». **Шум в категориях** — замените 5% значений на другие (`df.loc[mask, "destination"] = rng.choice(other_destinations, mask.sum())`) — имитирует ошибки ввода.

---

## 0.4. Готовые рецепты для датасетов курса

Ниже — компактные генераторы для файлов, которые встречаются в задачах. Каждый самодостаточен: скопируйте, запустите, получите CSV.

### `bookings.csv` — бронирования турагентства (задачи 1, 5, 20, 23, 58)

```python
def generate_bookings(n=1000, seed=42):
    rng = np.random.default_rng(seed)
    fake = Faker("ru_RU"); Faker.seed(seed)

    destinations = ["Турция", "Египет", "Италия", "Сочи", "Таиланд", "Греция"]
    dest_probs = [0.30, 0.20, 0.15, 0.15, 0.12, 0.08]

    df = pd.DataFrame({
        "booking_id": np.arange(100000, 100000 + n),
        "booking_date": pd.to_datetime(
            rng.integers(0, 730, n), unit="D", origin="2023-01-01"
        ),
        "destination": rng.choice(destinations, n, p=dest_probs),
        "office": rng.choice(["Офис1", "Офис2", "Офис3", "Офис4"], n),
        "tour_type": rng.choice(
            ["пляжный", "экскурсионный", "горнолыжный", "круиз", "сафари"],
            n, p=[0.40, 0.25, 0.15, 0.12, 0.08],
        ),
        "total_price": rng.lognormal(11.5, 0.5, n).round(-2).astype(int),
        "duration_days": rng.choice([3, 5, 7, 7, 7, 10, 10, 14], n),
        "status": rng.choice(
            ["подтверждён", "оплачен", "отменён", "в обработке"],
            n, p=[0.50, 0.35, 0.10, 0.05],
        ),
        "tourist_name": [fake.name() for _ in range(n)],
        "email": [fake.email() if rng.random() > 0.05 else None for _ in range(n)],
    })

    # 3 выброса (VIP-туры) — для задач 17, 27, 41
    outlier_idx = rng.choice(df.index, 3, replace=False)
    df.loc[outlier_idx, "total_price"] *= 20

    return df

# generate_bookings().to_csv("bookings.csv", index=False)
```

### `daily_revenue.csv` — дневная выручка (задачи 13, 21, 35, 50)

```python
def generate_daily_revenue(days=730, seed=42):
    rng = np.random.default_rng(seed)
    t = np.arange(days)
    dates = pd.date_range("2023-01-01", periods=days, freq="D")

    trend = 100_000 + 100 * t                       # рост 100 руб/день
    yearly = 30_000 * np.sin(2 * np.pi * t / 365)   # годовой цикл
    weekly = 15_000 * np.sin(2 * np.pi * t / 7)     # недельный цикл
    noise = rng.normal(0, 10_000, days)             # гауссов шум
    revenue = trend + yearly + weekly + noise

    # 5 выбросов (акции, праздники) — для задачи 13
    outlier_idx = rng.choice(days, 5, replace=False)
    revenue[outlier_idx] *= 2

    return pd.DataFrame({"date": dates, "revenue": revenue.round(2)})

# generate_daily_revenue().to_csv("daily_revenue.csv", index=False)
```

Три компоненты — тренд, годовая сезонность, недельная — дают реалистичный ряд, на котором видны **разные периоды** при разных окнах `rolling`: `rolling(7)` убирает недельную, но оставляет годовую; `rolling(30)` убирает обе, оставляет тренд. Это ровно то, что нужно для задачи 50.

### `cv_results.csv` — cross-validation (задачи 26, 41, 56)

```python
def generate_cv_results(seed=42):
    rng = np.random.default_rng(seed)
    base_acc = {"Logistic": 0.83, "RandomForest": 0.89,
                "SVM": 0.86, "XGBoost": 0.92, "NeuralNet": 0.89}
    noise = {"Logistic": 0.01, "RandomForest": 0.015,
             "SVM": 0.02, "XGBoost": 0.01, "NeuralNet": 0.03}

    rows = []
    for model, acc in base_acc.items():
        for fold in range(1, 6):
            rows.append({"model": model, "fold": fold,
                         "accuracy": round(acc + rng.normal(0, noise[model]), 4)})
    return pd.DataFrame(rows)

# generate_cv_results().to_csv("cv_results.csv", index=False)
```

Разный `noise` для разных моделей — **ключевая деталь**: `NeuralNet` имеет большой разброс, `Logistic` — маленький. На графике из задачи 26 это будет видно: у нейросети широкие планки ошибок, у логистической — узкие.

### `tourists.csv` — клиенты с целевой переменной (задачи 17, 38, 39, 41, 54, 60)

```python
def generate_tourists(n=5000, seed=42):
    rng = np.random.default_rng(seed)
    fake = Faker("ru_RU"); Faker.seed(seed)

    age = rng.integers(18, 75, n)
    income = rng.lognormal(10.5, 0.6, n).round(-3)
    avg_check = income * rng.uniform(0.02, 0.15, n) + rng.normal(0, 5000, n)
    duration = rng.choice([3, 5, 7, 10, 14], n)

    # Логистическая модель оттока: признаки коррелируют с целевой, но не детерминируют её
    logit = (-2.0 - 0.0001 * (income - income.mean()) / income.std()
             + 0.3 * (duration > 10) + 0.2 * (age < 25)
             + rng.normal(0, 0.5, n))
    prob = 1 / (1 + np.exp(-logit))
    churn = rng.binomial(1, prob * 0.15, n)        # 5–7% отток — дисбаланс

    return pd.DataFrame({
        "tourist_id": np.arange(10000, 10000 + n),
        "name": [fake.name() for _ in range(n)],
        "email": [fake.email() if rng.random() > 0.03 else None for _ in range(n)],
        "age": age, "income": income,
        "avg_check": avg_check.round(0),
        "duration_days": duration,
        "destination": rng.choice(["Турция", "Египет", "Италия", "Сочи", "Таиланд"], n),
        "gender": rng.choice(["М", "Ж"], n, p=[0.45, 0.55]),
        "churn": churn,
    })

# generate_tourists().to_csv("tourists.csv", index=False)
```

Здесь важна **логистическая модель** для целевой: `logit` — линейная комбинация признаков, `prob = sigmoid` — вероятность, `rng.binomial(1, p)` — бросок монеты. Умножение `prob * 0.15` снижает базовую вероятность до 5–7% — **дисбаланс**, который нужен для задачи 38. Признаки **коррелируют** с целевой (это видно на корреляционной матрице из задачи 29), но не детерминируют её — иначе модель была бы идеальной, и ROC-кривая шла бы через угол.

---

## 0.5. Чек-лист «прежде чем строить график»

1. **Есть ли датасет?** Проверьте Seaborn (`sns.load_dataset`), sklearn (`make_*`), Kaggle (`kagglehub`), Hugging Face (`load_dataset`).
2. **Если нет — сгенерируйте.** NumPy для чисел, Faker для строк, композиция для реализма.
3. **Зафиксируйте seed.** `rng = np.random.default_rng(42)` в начале.
4. **Добавьте шум и пропуски.** Без них данные выглядят «игрушечно», и задачи по очистке теряют смысл.
5. **Проверьте распределения.** `df.describe()`, `df["total_price"].hist()` — цены не должны быть отрицательными, возраст — в разумных пределах.
6. **Сохраните в CSV/Parquet.** `df.to_csv("bookings.csv", index=False)` — и используйте в задачах.
7. **Задокументируйте.** В `README` проекта — откуда данные, какой seed, какие допущения.

---

**Что дальше.** В следующем разделе мы разберём **основы Matplotlib** — от первого графика `plt.plot()` до настройки фигуры, осей и сохранения результата. Все примеры будут использовать данные, полученные одним из способов выше: если у вас нет `bookings.csv` — сгенерируйте его через `generate_bookings()` или скачайте с Kaggle через `kagglehub.dataset_download`. Если хотите — могу дописать **pytest-тесты** для генераторов (проверить, что `churn.mean() < 0.1`, что цены неотрицательны, что даты в правильном диапазоне) или **готовые ячейки-загрузчики** для Kaggle/Hugging Face с обработкой ошибок авторизации.